<div style="font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif; background: linear-gradient(135deg, #1e1e2f, #2a2a40); padding: 30px; border-radius: 12px; color: white; box-shadow: 0 4px 15px rgba(0,0,0,0.3);">
  <h1 style="margin-top: 0; color: #00e676; letter-spacing: 1px;">🎬 LTX-2.3 Video Generator</h1>
  <p style="font-size: 16px; line-height: 1.6; color: #d1d1e0;">
    Generate high-quality <b>Text-to-Video</b> and <b>Image-to-Video</b> directly in your browser.<br>
    This notebook uses a robust web interface, making it incredibly easy for beginners to use.
  </p>
  <hr style="border: 1px solid #4a4a6a; margin: 20px 0;">
  <h3 style="color: #b3b3ff; margin-bottom: 10px;">💠 Quick Start Instructions:</h3>
  <ol style="font-size: 15px; line-height: 1.8; color: #e6e6f2;">
    <li><b style="color:#ff5252;">CRITICAL:</b> Go to your Compute Profile (left sidebar) and select the <b>A100 GPU</b>. <i>(Do not run this on a CPU or it will crash!)</i></li>
    <li>Run <b>Step 1</b> below to install the AI engine and download the model weights (this takes a few minutes).</li>
    <li>Run <b>Step 2</b> to launch the interactive web interface, where you can type prompts and generate videos.</li>
  </ol>
</div>

## ⚙️ Step 1 — System Initialization & Weight Downloads
Run this **once** per session. It installs the backend engine and downloads the heavy LTX-2.3 model weights to your GPU container.

In [ ]:
import os, subprocess, time
from pathlib import Path
from IPython.display import display, HTML, clear_output

COMFY_PATH = "/root/ComfyUI"

display(HTML("<p style='color:#00e676;font-weight:bold;font-family:sans-serif'>[1/4] Installing PyTorch + dependencies...</p>"))
subprocess.run(["pip", "install", "-q", "torch>=2.4.0", "torchvision", "torchaudio", "--index-url", "https://download.pytorch.org/whl/cu121"], check=True)
subprocess.run(["pip", "install", "-q", "torchsde", "einops", "diffusers", "accelerate", "av", "spandrel", "albumentations", "onnx", "opencv-python-headless", "onnxruntime-gpu", "tqdm", "Pillow", "requests", "gradio"], check=True)

display(HTML("<p style='color:#00e676;font-weight:bold;font-family:sans-serif'>[2/4] Setting up Backend Engine...</p>"))
if not os.path.exists(COMFY_PATH):
    subprocess.run(["git", "clone", "-q", "https://github.com/comfyanonymous/ComfyUI", COMFY_PATH], check=True)
subprocess.run(["pip", "install", "-q", "-r", f"{COMFY_PATH}/requirements.txt"], check=True)

display(HTML("<p style='color:#00e676;font-weight:bold;font-family:sans-serif'>[3/4] Installing custom nodes...</p>"))
nodes_dir = f"{COMFY_PATH}/custom_nodes"
for node_url in [
    "https://github.com/kijai/ComfyUI-KJNodes",
    "https://github.com/city96/ComfyUI-GGUF",
    "https://github.com/Lightricks/ComfyUI-LTXVideo/",
    "https://github.com/Kosinkadink/ComfyUI-VideoHelperSuite",
    "https://github.com/kijai/ComfyUI-MelBandRoFormer"
]:
    name = node_url.rstrip("/").split("/")[-1]
    path = os.path.join(nodes_dir, name)
    if not os.path.exists(path):
        subprocess.run(["git", "clone", "-q", node_url, path], check=True)
        req = f"{path}/requirements.txt"
        if os.path.exists(req):
            subprocess.run(["pip", "install", "-q", "-r", req], check=True)

display(HTML("<p style='color:#00e676;font-weight:bold;font-family:sans-serif'>[4/4] Downloading LTX-2.3 weights...</p>"))
subprocess.run(["apt-get", "install", "-y", "-qq", "aria2"], check=True)

def dl(url, dest, fname):
    Path(dest).mkdir(parents=True, exist_ok=True)
    fpath = os.path.join(dest, fname)
    if not os.path.exists(fpath):
        subprocess.run(["aria2c", "--console-log-level=error", "-c", "-x", "16", "-s", "16", "-k", "1M", "-d", dest, "-o", fname, url], check=True)

B = COMFY_PATH + "/models"
dl("https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/ltx-2.3-22b-dev-Q4_K_M.gguf", f"{B}/unet", "ltx-2.3-22b-dev-Q4_K_M.gguf")
dl("https://huggingface.co/Dampfinchen/google-gemma-3-12b-it-qat-q4_0-gguf-small-fix/resolve/main/gemma-3-12b-it-q4_0_s.gguf", f"{B}/text_encoders", "gemma-3-12b-it-q4_0_s.gguf")
dl("https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/text_encoders/ltx-2.3-22b-dev_embeddings_connectors.safetensors", f"{B}/text_encoders", "ltx-2.3-22b-dev_embeddings_connectors.safetensors")
dl("https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_video_vae.safetensors", f"{B}/vae", "ltx-2.3-22b-dev_video_vae.safetensors")
dl("https://huggingface.co/unsloth/LTX-2.3-GGUF/resolve/main/vae/ltx-2.3-22b-dev_audio_vae.safetensors", f"{B}/vae", "ltx-2.3-22b-dev_audio_vae.safetensors")
dl("https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-spatial-upscaler-x2-1.0.safetensors", f"{B}/latent_upscale_models", "ltx-2.3-spatial-upscaler-x2-1.0.safetensors")
dl("https://huggingface.co/Kijai/MelBandRoFormer_comfy/resolve/main/MelBandRoformer_fp16.safetensors", f"{B}/diffusion_models", "MelBandRoformer_fp16.safetensors")
dl("https://huggingface.co/Kijai/LTX2.3_comfy/resolve/main/vae/taeltx2_3.safetensors", f"{B}/vae", "taeltx2_3.safetensors")
dl("https://huggingface.co/Lightricks/LTX-2.3/resolve/main/ltx-2.3-22b-distilled-lora-384.safetensors", f"{B}/loras", "ltx-2.3-22b-distilled-lora-384.safetensors")

clear_output()
display(HTML("<div style='padding:15px;background:#e8f5e9;border-left:5px solid #4caf50;border-radius:4px;color:#2e7d32;font-family:sans-serif;'><b>✨ Initialization Complete!</b> The GPU is ready. Proceed to Step 2 to launch the UI.</div>"))


## 🚀 Step 2 — Launch the Video Generator UI
Run this cell to open the interactive web interface. You can type prompts, upload images, and generate your videos directly from the UI without touching any code.
*(A public sharing link will also be generated at the bottom!)*

In [ ]:
import gradio as gr
import json, urllib.request, urllib.error, time, os, glob, socket, shutil, subprocess
from PIL import Image

COMFY_PATH  = "/root/ComfyUI"
OUTPUT_PATH = f"{COMFY_PATH}/output"
INPUT_PATH  = f"{COMFY_PATH}/input"

def is_server_running(port=8188):
    with socket.socket(socket.AF_INET, socket.SOCK_STREAM) as s:
        return s.connect_ex(("127.0.0.1", port)) == 0

def boot_server():
    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
    subprocess.Popen(["python", "main.py", "--dont-print-server"], cwd=COMFY_PATH)
    start_time = time.time()
    while not is_server_running():
        if time.time() - start_time > 120:
            raise RuntimeError("Backend server failed to start within 2 minutes. Check ComfyUI logs.")
        time.sleep(2)

def load_workflow():
    url = "https://raw.githubusercontent.com/AICHUCKY/Comfyui-Workflows/AICHUCKY-patch-1/Ltx2.3%20.json"
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r:
        return json.loads(r.read())

def queue_prompt(wf):
    data = json.dumps({"prompt": wf}).encode("utf-8")
    req  = urllib.request.Request("http://127.0.0.1:8188/prompt", data=data)
    try:
        return json.loads(urllib.request.urlopen(req).read())
    except urllib.error.HTTPError as e:
        raise RuntimeError(f"API ERROR: {e.read().decode()}")

def get_latest_video():
    mp4s = glob.glob(f"{OUTPUT_PATH}/**/*.mp4", recursive=True) + glob.glob(f"{OUTPUT_PATH}/*.mp4")
    if not mp4s:
        return None
    return max(mp4s, key=os.path.getctime)

def generate_video(mode, image_filepath, prompt, width, height, duration, seed, progress=gr.Progress()):
    progress(0, desc="Starting Server...")
    if not os.path.exists(COMFY_PATH):
        raise gr.Error("Engine not found! You must run Step 1 and use an A100 GPU before generating.")
    if not is_server_running():
        boot_server()
    os.makedirs(INPUT_PATH, exist_ok=True)
    
    W = max(256, round(width / 32) * 32)
    H = max(256, round(height / 32) * 32)
    wf = load_workflow()
    
    wf["292"]["inputs"]["value"] = W
    wf["293"]["inputs"]["value"] = H
    wf["285"]["inputs"]["value"] = 24
    wf["121"]["inputs"]["text"] = prompt
    wf["291"]["inputs"]["value"] = duration
    wf["137"]["inputs"]["sampler_name"] = "lcm"
    wf["360"]["inputs"]["sigmas"] = "1.0, 0.99375, 0.9875, 0.98125, 0.975, 0.909375, 0.725, 0.421875, 0.0"
    wf["129"]["inputs"]["cfg"] = 1.0
    wf["115"]["inputs"]["noise_seed"] = seed
    wf["138"]["inputs"]["sampler_name"] = "euler_cfg_pp"
    wf["359"]["inputs"]["sigmas"] = "0.85, 0.7250, 0.4219, 0.0"
    wf["103"]["inputs"]["cfg"] = 1.0
    wf["114"]["inputs"]["noise_seed"] = seed + 1

    progress(0.1, desc="Preparing Inputs...")
    if mode == "Text-to-Video":
        wf["290"]["inputs"]["value"] = True
        dummy_name = "dummy_t2v.jpg"
        Image.new("RGB", (W, H), "black").save(os.path.join(INPUT_PATH, dummy_name))
        wf["167"]["inputs"]["image"] = dummy_name
    else:
        if image_filepath is None:
            raise gr.Error("Please upload an image for Image-to-Video.")
        wf["290"]["inputs"]["value"] = False
        filename = os.path.basename(image_filepath)
        shutil.copy(image_filepath, os.path.join(INPUT_PATH, filename))
        wf["167"]["inputs"]["image"] = filename

    progress(0.2, desc="Queuing Generation...")
    pid = queue_prompt(wf)["prompt_id"]
    progress(0.3, desc="Rendering Video (This takes time)...")
    
    while True:
        try:
            h = json.loads(urllib.request.urlopen(f"http://127.0.0.1:8188/history/{pid}").read())
            if str(pid) in h: 
                break
            q = json.loads(urllib.request.urlopen("http://127.0.0.1:8188/queue").read())
            if not any(str(j[1])==str(pid) for j in q.get("queue_running",[])+q.get("queue_pending",[])):
                raise gr.Error("Generation failed or crashed.")
        except Exception as e:
            if "Generation failed" in str(e): raise e
            pass
        time.sleep(3)
        
    progress(1.0, desc="Done!")
    return get_latest_video()

with gr.Blocks(theme=gr.themes.Monochrome()) as demo:
    gr.Markdown("# 🎬 LTX-2.3 Video Generator")
    gr.Markdown("Select your mode below, adjust your settings, and generate!")
    with gr.Row():
        with gr.Column(scale=1):
            mode_selector = gr.Radio(["Text-to-Video", "Image-to-Video"], value="Text-to-Video", label="Generation Mode")
            image_input = gr.Image(type="filepath", label="Upload Starting Image", visible=False)
            prompt_input = gr.Textbox(label="Prompt", placeholder="A cinematic shot...", lines=3)
            with gr.Row():
                width_slider = gr.Slider(minimum=256, maximum=1920, step=32, value=832, label="Width")
                height_slider = gr.Slider(minimum=256, maximum=1080, step=32, value=480, label="Height")
            with gr.Row():
                duration_slider = gr.Slider(minimum=1, maximum=10, step=1, value=3, label="Duration (s)")
                seed_input = gr.Number(value=43, label="Seed", precision=0)
            generate_btn = gr.Button("▶ Generate Video", variant="primary")
        with gr.Column(scale=1):
            video_output = gr.Video(label="Generated Output")
            
    def update_visibility(mode):
        return gr.update(visible=(mode == "Image-to-Video"))
    mode_selector.change(fn=update_visibility, inputs=mode_selector, outputs=image_input)
    generate_btn.click(fn=generate_video, inputs=[mode_selector, image_input, prompt_input, width_slider, height_slider, duration_slider, seed_input], outputs=video_output)

demo.launch(share=True, inline=True)
